# Exploratory Data Analysis (EDA) / Research

This notebook is intended for deep data exploration and feature analysis on Kaggle or locally. It uses the shared Kaggle boilerplate to ensure paths and dependencies are correctly loaded.

In [ ]:
# Removed autoreload to prevent 'imp' module error in Python 3.12

In [ ]:
!pip install -q optuna pytorch-lightning omegaconf catboost

In [ ]:
!ls -laR /kaggle/input || true
!pwd

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import torch

warnings.filterwarnings('ignore')

# Fix for Kaggle P100 GPU PyTorch LSTM bug (CUDA error: no kernel image is available)
torch.backends.cudnn.enabled = False

# Detect if running on Kaggle
is_kaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if is_kaggle:
    # Auto-discover the code dataset path
    input_dir = Path('/kaggle/input')
    code_dataset_path = None
    for p in input_dir.rglob('conf/config.yaml'):
        code_dataset_path = p.parents[1]
        break
        
    if code_dataset_path is None:
        raise FileNotFoundError("Could not find conf/config.yaml inside /kaggle/input")
        
    CODE_DATASET_PATH = code_dataset_path
    if str(CODE_DATASET_PATH) not in sys.path:
        sys.path.append(str(CODE_DATASET_PATH))
    # Kaggle default working directory is /kaggle/working
    PROJECT_ROOT = Path('/kaggle/working')
    print(f"Running on Kaggle. Added {CODE_DATASET_PATH} to sys.path")
else:
    # If running locally, make sure the project root is in sys.path
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)
    print(f"Running locally. Changed working directory to {PROJECT_ROOT}")

In [ ]:
import yaml

# Load Config
conf_path = Path('conf/config.yaml')
if is_kaggle:
    conf_path = CODE_DATASET_PATH / 'conf/config.yaml'

with open(conf_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Override environment if running on Kaggle
if is_kaggle:
    config['env'] = 'kaggle'
    
    # Auto-discover data_dir
    data_dir = None
    for p in Path('/kaggle/input').rglob('ch2026_metrics_train.csv'):
        data_dir = p.parent
        break
        
    if data_dir is not None:
        config['paths']['kaggle']['data_dir'] = str(data_dir)
        print(f"Auto-discovered data_dir: {data_dir}")

env = config['env']
paths = config['paths'][env]

print(f"Environment: {env}")
print(f"Data Dir: {paths['data_dir']}")
print(f"Model Dir: {paths['model_dir']}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.data_loader import DataLoader
from src.data.make_dataset import build_training_dataset
from src.features.build_features import engineer_features

# Load your datasets here
print("Loading data via standard DataLoader...")
data_loader = DataLoader(data_dir=paths['data_dir'])
metrics_df = data_loader.load_metrics()
sensor_data_dict = data_loader.load_sensor_data()

print("Building base dataset...")
df_train_raw = build_training_dataset(metrics_df, sensor_data_dict)

print("Engineering features...")
df_train = engineer_features(df_train_raw)
print(f"Data shape: {df_train.shape}")


## Basic Statistics

In [ ]:
df_train.info()
display(df_train.describe())

## Target Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_train[config['training']['target_col']], bins=50, kde=True)
plt.title('Target Distribution')
plt.show()